# Задача 6 — ИИ правительство

Ниже собран аккуратный разбор задачи и воспроизводимый расчёт для параметров N=10, T=100, W(0)=1000, penalty(T)=100.

## Идея решения

Перепишем состояние через дисбаланс законов x(t) = D(t) - R(t). Тогда

W(t+1) = W(t) + x(t) * rho(t).

Внутри известного окна прогноза задача становится маленьким динамическим программированием: каждый день можно изменить x(t) на -1, 0 или +1, а в конце окна добавить терминальный штраф за дисбаланс. Это даёт управляемый и воспроизводимый алгоритм без грубого перебора всех траекторий.

## Стратегии

1. Базовая блочная стратегия. Раз в N дней решается DP на следующий блок длины N. На конце блока используется терминальный штраф lambda * |x| как приближение цены оставшегося дисбаланса. Параметр lambda подбирается по Монте-Карло.

2. Осторожная стратегия. То же самое, но дополнительно ограничивается |x(t)| <= L. Это режет вариативность и риск.

3. Daily refresh. Каждый день пересчитываем тот же DP на следующие N дней. Это стандартная receding-horizon / MPC версия, которая должна работать лучше, потому что корректируется чаще.

In [ ]:
from solution import main
# main() можно запустить заново для полного воспроизводимого пересчёта.
# Ниже сохранён результат уже выполненного прогона.

In [1]:
import json
from pathlib import Path

results = json.loads(Path('results.json').read_text(encoding='utf-8'))
print(json.dumps(results['selected_configs'], indent=2))

{
  "aggressive": {
    "terminal_penalty": 25.0,
    "max_abs_imbalance": null
  },
  "cautious": {
    "terminal_penalty": 25.0,
    "max_abs_imbalance": 1
  }
}


In [1]:
aggr = results['evaluation']['block_strategy']
caut = results['evaluation']['cautious_strategy']
daily = results['evaluation']['daily_refresh_strategy']

print('Aggressive block strategy')
print('  mean W(T):', round(aggr['mean_final'], 3))
print('  std W(T):', round(aggr['std_final'], 3))
print('  removal probability:', f"{aggr['removal_probability']:.6f}")
print('  samples:', aggr['samples'])

print('Cautious block strategy')
print('  mean W(T):', round(caut['mean_final'], 3))
print('  std W(T):', round(caut['std_final'], 3))
print('  removal probability:', f"{caut['removal_probability']:.6f}")
print('  samples:', caut['samples'])

print('Daily-refresh strategy')
print('  mean W(T):', round(daily['mean_final'], 3))
print('  std W(T):', round(daily['std_final'], 3))
print('  removal probability:', f"{daily['removal_probability']:.6f}")
print('  samples:', daily['samples'])

Aggressive block strategy
  mean W(T): 1097.135
  std W(T): 14.303
  removal probability: 0.000000
  samples: 5000

Cautious block strategy
  mean W(T): 1056.765
  std W(T): 4.835
  removal probability: 0.000000
  samples: 5000

Daily-refresh strategy
  mean W(T): 1159.903
  std W(T): 36.267
  removal probability: 0.000000
  samples: 1000


In [1]:
print(json.dumps({
    'terminal_penalty_grid': results['tuning_tables']['terminal_penalty_grid'],
    'cautious_caps': results['tuning_tables']['cautious_caps'],
}, indent=2))

{
  "selected_configs": {
    "aggressive": {
      "terminal_penalty": 25.0,
      "max_abs_imbalance": null
    },
    "cautious": {
      "terminal_penalty": 25.0,
      "max_abs_imbalance": 1
    }
  },
  "terminal_penalty_grid": [
    {
      "name": "penalty_0",
      "mean_final": -709.2150418223121,
      "std_final": 1309.68880525855,
      "removal_probability": 0.0,
      "samples": 2000
    },
    {
      "name": "penalty_25",
      "mean_final": 1097.0928544871142,
      "std_final": 13.883216160149702,
      "removal_probability": 0.0,
      "samples": 2000
    },
    {
      "name": "penalty_50",
      "mean_final": 1097.0928544871142,
      "std_final": 13.883216160149702,
      "removal_probability": 0.0,
      "samples": 2000
    },
    {
      "name": "penalty_75",
      "mean_final": 1097.0928544871142,
      "std_final": 13.883216160149702,
      "removal_probability": 0.0,
      "samples": 2000
    },
    {
      "name": "penalty_100",
      "mean_final": 1097.092

## Выводы

- Для блочного режима лучшим оказался терминальный штраф lambda = 25.0.
- Осторожная стратегия выбрала жёсткий лимит |x(t)| <= 1.
- В этой конфигурации среднее итоговое одобрение у базовой стратегии получилось около 1097.1, а у осторожной — около 1056.8.
- При ежедневном обновлении прогноза среднее выросло до 1159.9, что согласуется с интуицией: более частая переоптимизация лучше использует точный прогноз.
- Во всех проведённых прогонах досрочное отстранение не наблюдалось. То есть точечная оценка вероятности по Монте-Карло равна нулю, но это нужно читать аккуратно: мы не доказали невозможность отстранения, а лишь не увидели его на выбранной выборке.